# Varhadi voice: fine-tune MMS-TTS Marathi on narrator clips (Luke)
PRIVATE RESEARCH USE ONLY. Source recordings have an unverified licence and a real narrator's voice; never publish the result. Runtime: T4 GPU. Upload `luk_part0.zip`, `luk_part1.zip`, `luk_part2.zip` to /content first. Run the cells in order and **keep this tab in front** while training (about 75 min): Colab throttles and recycles background tabs, which lost a finished model once. Recipe: github.com/ylacombe/finetune-hf-vits (MMS checkpoints are CC BY-NC 4.0).

In [ ]:
!nvidia-smi -L
!git clone -q https://github.com/ylacombe/finetune-hf-vits.git /content/finetune-hf-vits
%cd /content/finetune-hf-vits
!git checkout -q 6f3f51f4d667f5c3eef89484d151ffd39d2c2b89
!pip -q install -r requirements.txt
# the 2024 repo needs the transformers 4.x VitsConfig (v5 dropped pad_token_id); 4.46.3 also runs on Colab's Python 3.13
!pip -q install "transformers==4.46.3"
# the script reads batch["speaker_id"] even for a single-speaker dataset
!sed -i 's/speaker_id=batch\["speaker_id"\]/speaker_id=batch.get("speaker_id")/' run_vits_finetuning.py
# matplotlib 3.10 removed tostring_rgb (only used to draw alignment plots at eval time)
!sed -i 's/np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)/np.ascontiguousarray(np.asarray(fig.canvas.buffer_rgba())[..., :3]).reshape(-1)/' utils/plot.py
!cd monotonic_align && mkdir -p monotonic_align && python setup.py build_ext --inplace

In [ ]:
# approve the Google permission popup NOW, so the finished voice can be copied to Drive at the end
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import glob, zipfile, os
os.makedirs('/content/luke_ds', exist_ok=True)
for z in sorted(glob.glob('/content/luk_part*.zip')):
    zipfile.ZipFile(z).extractall('/content/luke_ds')
print(len(glob.glob('/content/luke_ds/wavs/*')), 'clips;', open('/content/luke_ds/metadata.csv', encoding='utf-8').read()[:300])

In [ ]:
%cd /content/finetune-hf-vits
!python convert_original_discriminator_checkpoint.py --language_code mar --pytorch_dump_folder_path /content/mms-tts-mar-train
!ls -la /content/mms-tts-mar-train

In [ ]:
import json
true, false = True, False
cfg = {
 "project_name": "varhadi_tts_luke",
 "push_to_hub": false,
 "output_dir": "/content/vits_luke",
 "model_name_or_path": "/content/mms-tts-mar-train",
 "dataset_name": "/content/luke_ds",
 "audio_column_name": "audio",
 "text_column_name": "text",
 "train_split_name": "train",
 "eval_split_name": "train",
 "max_eval_samples": 12,
 "max_duration_in_seconds": 20,
 "min_duration_in_seconds": 1.0,
 "max_tokens_length": 500,
 "do_lower_case": false,
 "preprocessing_num_workers": 1,
 "full_generation_sample_text": "तुमी पिकाले पाणी देऊन झालं का? काई अडचण असीन तर कृषी सहाय्यकाले सांगा.",
 "do_train": true,
 "do_eval": true,
 "num_train_epochs": 100,
 "per_device_train_batch_size": 8,
 "per_device_eval_batch_size": 8,
 "learning_rate": 2e-05,
 "adam_beta1": 0.8,
 "adam_beta2": 0.99,
 "fp16": true,
 "seed": 456,
 "eval_steps": 500,
 "save_steps": 500,
 "save_total_limit": 2,
 "logging_steps": 50,
 "report_to": "none",
 "weight_disc": 3,
 "weight_fmaps": 1,
 "weight_gen": 1,
 "weight_kl": 1.5,
 "weight_duration": 1,
 "weight_mel": 35
}
json.dump(cfg, open('/content/finetune_luke.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('config written')

In [ ]:
%cd /content/finetune-hf-vits
# foreground run (keeps the session busy); ~1 step/s on a T4, 4400 steps. Only every 100th step and errors are printed.
!rm -rf /content/vits_luke
!accelerate launch run_vits_finetuning.py /content/finetune_luke.json 2>&1 | tr '\r' '\n' | grep --line-buffered -E "Steps: .*\| [0-9]*00/4400|Error|Traceback" | cut -c1-150

In [ ]:
import torch
from transformers import VitsModel, AutoTokenizer
m, info = VitsModel.from_pretrained('/content/vits_luke', output_loading_info=True)   # drops the training-only discriminator
print('missing keys:', info['missing_keys'], '| dropped:', len(info['unexpected_keys']))
tok = AutoTokenizer.from_pretrained('/content/vits_luke'); m.eval(); torch.manual_seed(0)
with torch.no_grad(): w = m(**tok('तुमी पिकाले पाणी देऊन झालं का', return_tensors='pt')).waveform[0]
print('test sentence: %.2f s, rms %.3f' % (w.shape[0] / m.config.sampling_rate, float(w.pow(2).mean().sqrt())))
m.save_pretrained('/content/varhadi_voice'); tok.save_pretrained('/content/varhadi_voice')
!cd /content/varhadi_voice && zip -q -r /content/varhadi_voice.zip . && mkdir -p /content/drive/MyDrive/KisaanDost && cp /content/varhadi_voice.zip /content/drive/MyDrive/KisaanDost/varhadi_voice.zip && ls -la /content/drive/MyDrive/KisaanDost/varhadi_voice.zip